# ChakraModel V3 Strict Verification - Anti-Fabrication Harness

This notebook integrates the Anti-Fabrication toolkit explicitly. It statically verifies the weights, uses strict constraints, and prints the exact outputs required by `verifyai`.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless pycocotools

In [ ]:
import os
import sys
import hashlib
from pathlib import Path

# Set Anti-Fabrication Toolkit Variables
os.environ["VERIFY_NONCE"] = "KAGGLE_V3_RUN_9901"

# EXPLICITLY define dataset root to avoid duplicates (BUG 2 FIX)
# Change this to match your actual Kaggle input path!
DATASET_ROOT = '/kaggle/input/polypdb-polyp-raw/PolypDB'
DATASET_NAME = 'PolypDB'

# Find the code directory strictly
code_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'chakranet_segmenter.py' in files and 'src' in root:
        code_dir = root
        break

if not code_dir:
    # Fallback without 'src' check
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files:
            code_dir = root
            break

assert code_dir is not None, "FATAL: Code directory not found. Did you attach the ChakraModel_Kaggle_Code dataset?"
sys.path.insert(0, code_dir)
print(f"Codebase verified at: {code_dir}")

In [ ]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
from chakranet_segmenter import ChakraNet
from metrics_engine_v2 import MetricsEngineV2

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

yolo_path = None
vit_path = None

# STRICT weight path finding to avoid duplicate best.pt files (BUG 1 FIX)
for root, dirs, files in os.walk('/kaggle/input'):
    if 'transformer' in root.lower() or 'weight' in root.lower():
        for f in files:
            if f == 'best.pt': yolo_path = os.path.join(root, f)
            if f == 'chakra_transformer_best.pth': vit_path = os.path.join(root, f)

assert yolo_path is not None, "FATAL: YOLO best.pt not found in a valid weights folder!"
assert vit_path is not None, "FATAL: chakra_transformer_best.pth not found!"

print("=" * 60)
print("STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION (V3)")
print("=" * 60)
print(f"VERIFY_NONCE: {os.environ['VERIFY_NONCE']}")
print(f"Segmenter Checkpoint: {vit_path}")
print(f"Segmenter MD5: {md5(vit_path)}")
print(f"YOLO Checkpoint: {yolo_path}")
print(f"YOLO MD5: {md5(yolo_path)}")
print("=" * 60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo_model = YOLO(yolo_path)
yolo_model.to(device)

segmenter = ChakraNet(img_size=(384, 384), device=device)
segmenter.model.to(device)

sd = torch.load(vit_path, map_location=device)
sd_fixed = {k.replace('_orig_mod.', '').replace('module.', ''): v for k, v in sd.items()}
res = segmenter.model.load_state_dict(sd_fixed, strict=False)

print(f"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}")

# STRICT BUG FIX #3: Crash if weights are severely mismatched
if len(res.missing_keys) > 50:
    print(f"[FATAL] Major weight mismatch! {len(res.missing_keys)} keys missing. The loaded model is invalid.")
    sys.exit(1)

In [ ]:
print(f"\n============================================================")
print(f"VERIFYING DATASET: {DATASET_NAME}")
print(f"============================================================")

d_root = Path(DATASET_ROOT)
images_dir = d_root / "images"
masks_dir = d_root / "masks"

if not images_dir.exists() or not masks_dir.exists():
    print(f"[FATAL] Images or masks directory not found at {DATASET_ROOT}")
    print("HINT: Double check the path in Kaggle's 'Input' sidebar. It might be nested (e.g. /kaggle/input/polypdb-polyp-raw/PolypDB/images)")
    sys.exit(1)

image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
print(f"Total images found: {len(image_paths)}")

metrics_engine = MetricsEngineV2()
results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}

for i, img_path in enumerate(image_paths):
    mask_path = None
    for ext in [".png", ".jpg", ".tif", ".bmp"]:
        potential_path = masks_dir / (img_path.stem + ext)
        if potential_path.exists():
            mask_path = potential_path
            break
            
    if not mask_path:
        print(f"[WARN] Mask not found for {img_path.name}")
        continue
        
    img_bgr = cv2.imread(str(img_path))
    gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    img_resized = cv2.resize(img_bgr, (448, 448))
    yolo_results = yolo_model(img_resized, verbose=False)[0]
    c_mask = np.zeros((448, 448), dtype=np.uint8)
    
    if len(yolo_results.boxes) > 0:
        boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
        box = boxes[0].xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, box)
        
        w_box, h_box = x2 - x1, y2 - y1
        px, py = int(0.25 * w_box), int(0.25 * h_box)
        px1, py1 = max(0, x1 - px), max(0, y1 - py)
        px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))
        
        if px2 > px1 and py2 > py1:
            roi_crop = img_resized[py1:py2, px1:px2]
            seg_roi_mask, _, _, _ = segmenter.segment_roi(roi_crop, threshold=0.45)
            if seg_roi_mask is not None:
                seg_roi_mask = cv2.resize(seg_roi_mask, (px2-px1, py2-py1), interpolation=cv2.INTER_NEAREST)
                c_mask[py1:py2, px1:px2] = seg_roi_mask
                
    gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
    
    c_bin = (c_mask > 127).astype(np.uint8)
    gt_bin = (gt_resized > 127).astype(np.uint8)
    
    metrics = metrics_engine.compute_all(c_bin, gt_bin)
    for k in results.keys():
        results[k].append(metrics[k])
    
    if (i + 1) % 50 == 0 or (i + 1) == len(image_paths):
        print(f"[{i + 1}/{len(image_paths)}] Processed. Current Avg Dice: {np.mean(results['Dice']):.4f}")

print(f"\n[DONE] {DATASET_NAME}")
print(f"Total Evaluated Images: {len(results['Dice'])}")
for k in results.keys():
    print(f"Final True Average {k}: [CANARY] {float(np.mean(results[k])):.4f}")
print(f"============================================================")